# Wi-Fi CSI presence detection — Stage 1 data analysis

Room4 of the IEEE DataPort *CSI Dataset for WiFi Based Human Detection* (DOI 10.21227/smn7-jb67):
empty room (label 0) vs one person present (label 1).
Each magnitude CSV = one ~70 s recording; rows = packets (time), 49 columns = subcarriers; no header, no timestamps.

## 0. Setup and data location
Works in two places without edits:
- **JupyterHub:** upload `data_room4.zip` next to this notebook; the cell below unzips it once.
- **Local repo:** uses the project's `CSV Files/` folder one level up.

In [ ]:
import glob
import os
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# find the data: next to the notebook (JupyterHub) or one level up (local repo)
if not os.path.isdir("CSV Files") and os.path.isfile("data_room4.zip"):
    with zipfile.ZipFile("data_room4.zip") as z:
        z.extractall(".")
DATA = "CSV Files" if os.path.isdir("CSV Files") else "../CSV Files"
assert os.path.isdir(DATA), "CSV Files/ not found - upload data_room4.zip next to this notebook"

# magnitude files only: the csi_complex_data/ folders (amplitude + phase) are not used,
# so CSI phase is never loaded (dropped by design)
FOLDERS = {
    "empty":      f"{DATA}/room4_iqbal/Static_noperson_channel_11_20mhz/csi_magnitude_data",
    "person":     f"{DATA}/room4_iqbal/Dynamic_1_person_ch11_20mhz/csi_magnitude_data",
    # not used for training: only checked to decide what "Static" means (optional folder)
    "lab_static": f"{DATA}/LAB_Final Year Project/Static_Lab_ch6_20mhz/csi_magnitude_data",
}
DROP_SUBCARRIERS = [0, 48]   # band edges: 1.3-7x noisier than inner subcarriers in every empty file
FILE_S = 70    # seconds between consecutive file start times (from the filenames)
WIN_S = 2      # window length in seconds
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

## Helpers

In [ ]:
def list_csv(folder):
    return sorted(glob.glob(folder + "/*.csv"))


def load(path):
    df = pd.read_csv(path, header=None)       # no header row: the first line is data
    return df.drop(columns=DROP_SUBCARRIERS)  # 49 -> 47 subcarriers (original indices kept)


def spread(df):
    """Median over subcarriers of the per-subcarrier IQR. Robust to the rare amplitude spikes."""
    iqr = df.quantile(0.75) - df.quantile(0.25)   # one number per subcarrier
    return iqr.median()


def window(df, start_s, length_s=WIN_S, file_s=FILE_S):
    """Cut a length_s-second window starting at start_s, using this file's own packet rate."""
    rate = len(df) / file_s                      # estimated packets per second for this file
    a = int(start_s * rate)
    b = a + int(length_s * rate)
    return df.iloc[a:b].to_numpy()               # shape: (packets, 47)


def summarise_folder(files):
    """Load each file ONCE and return its packet count and spread."""
    packets, spreads = [], []
    for f in files:
        df = load(f)
        packets.append(len(df))
        spreads.append(spread(df))
    return packets, spreads


def start_time(path):
    """Recording start time from a filename like csi_20241204_020632_magnitude.csv."""
    stamp = os.path.basename(path).split("_")[1:3]            # ['20241204', '020632']
    return pd.to_datetime("_".join(stamp), format="%Y%m%d_%H%M%S")


def split_files(files, n_train, n_val):
    return {"train": files[:n_train],
            "val":   files[n_train:n_train + n_val],
            "test":  files[n_train + n_val:]}

## 1. Inspect one file per class

In [ ]:
files = {name: list_csv(folder) for name, folder in FOLDERS.items()}
first = {name: load(files[name][0]) for name in ("empty", "person")}

for name, df in first.items():
    print(f"\n=== {name}: {len(files[name])} files, first = {os.path.basename(files[name][0])}")
    print("shape (packets, subcarriers):", df.shape)
    print("missing values:", df.isna().sum().sum())
    display(pd.DataFrame({"std": df.std(), "IQR": df.quantile(0.75) - df.quantile(0.25)}).T.round(1))

## 2. Raw signal of one subcarrier over a whole recording

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharey=True, constrained_layout=True)
for ax, name, title in zip(axes, ("empty", "person"),
                           ("room4, empty (Static_noperson)", "room4, 1 person moving (Dynamic_1_person)")):
    ax.plot(first[name][1])
    ax.set_title(f"{title} - subcarrier 1")
    ax.set_xlabel("packet index")
    ax.set_ylabel("amplitude")
plt.savefig(f"{FIG_DIR}/fig_raw_subcarrier1.png", dpi=150)
plt.show()

## 3. Spread per recording, every file

In [ ]:
summary = {name: summarise_folder(files[name]) for name in FOLDERS if files[name]}

for name, (packets, spreads) in summary.items():
    print(f"{name:10s} files={len(spreads):3d}  median IQR min={min(spreads):6.1f}  max={max(spreads):6.1f}")
# label check: lab_static lands inside the person range, so "Static" is ambiguous -> excluded

## 4. Dataset statistics (room4 only)

In [ ]:
rows = []
for name, label in (("empty", 0), ("person", 1)):
    packets, spreads = summary[name]
    rows.append({
        "class": name,
        "label": label,
        "files": len(packets),
        "packets (total)": sum(packets),
        "duration (s, ≈)": len(packets) * FILE_S,
        "packets/s (median)": round(np.median(packets) / FILE_S),
        f"{WIN_S}-s windows (non-overlapping)": len(packets) * (FILE_S // WIN_S),
        "median IQR (min–max)": f"{min(spreads):.1f}–{max(spreads):.1f}",
    })
table = pd.DataFrame(rows)
table.to_csv(f"{FIG_DIR}/table_dataset_stats.csv", index=False)
table

## 5. Train / validation / test split
Chronological, per class, by **file** (never by window): earliest 60% train, next 20% validation, latest 20% test.

In [ ]:
splits = {
    "empty":  split_files(files["empty"],  n_train=6,  n_val=2),    # 6 / 2 / 2
    "person": split_files(files["person"], n_train=51, n_val=17),   # 51 / 17 / 17
}

split_rows = []
for name, parts in splits.items():
    for part, part_files in parts.items():
        split_rows.append({
            "class": name,
            "set": part,
            "files": len(part_files),
            f"{WIN_S}-s windows": len(part_files) * (FILE_S // WIN_S),
            "first start": start_time(part_files[0]),
            "last start": start_time(part_files[-1]),
        })
split_table = pd.DataFrame(split_rows)
split_table.to_csv(f"{FIG_DIR}/table_split.csv", index=False)

# no file may appear in two sets
all_split = [f for parts in splits.values() for part_files in parts.values() for f in part_files]
assert len(all_split) == len(set(all_split)) == len(files["empty"]) + len(files["person"])
split_table

## Figure (a): one 2-s window per class

In [ ]:
centred = {}
for name in ("empty", "person"):
    w = window(first[name], start_s=10)
    centred[name] = w - np.median(w, axis=0)     # shows VARIATION, not absolute level
lim = np.percentile(np.abs(np.vstack(list(centred.values()))), 99)   # shared colour range, ignores spikes

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True, constrained_layout=True)
for ax, name, title in zip(axes, ("empty", "person"),
                           ("Empty room (label 0)", "One person present (label 1)")):
    c = centred[name]
    im = ax.imshow(c.T, aspect="auto", origin="lower", cmap="RdBu_r", vmin=-lim, vmax=lim,
                   extent=[0, WIN_S, 1, 48])
    ax.set_title(title)
    ax.set_xlabel("time in window (s)")
axes[0].set_ylabel("subcarrier index (0 and 48 dropped)")
fig.colorbar(im, ax=axes, label="amplitude − subcarrier median")
plt.savefig(f"{FIG_DIR}/fig_a_heatmap.png", dpi=150)
plt.show()

## Figure (b): spread per recording, empty vs person

In [ ]:
empty_spreads, person_spreads = summary["empty"][1], summary["person"][1]
fig, ax = plt.subplots(figsize=(5, 4), constrained_layout=True)
ax.boxplot([empty_spreads, person_spreads])
ax.set_xticks([1, 2], [f"Empty (0)\n{len(empty_spreads)} files", f"Person (1)\n{len(person_spreads)} files"])
ax.set_ylabel("median IQR across subcarriers")
ax.set_yscale("log")          # values span ~7 to ~300 - log scale keeps both boxes readable
ax.set_title("Typical amplitude spread per recording, room4")
plt.savefig(f"{FIG_DIR}/fig_b_spread_boxplot.png", dpi=150)
plt.show()